# NB2 — XGBoost walk-forward 2020–2025 → dự báo xu hướng (uptrend / sideway / downtrend)

Nhánh **AI** của luồng nghiên cứu, chạy **độc lập với NB1**. Tự đọc tệp OHLCV,
tự tính chỉ báo, tự gán nhãn, huấn luyện **XGBoost** theo **walk-forward**: mỗi năm
2020 → 2025 được dự báo bởi mô hình chỉ học dữ liệu **trước** năm đó. Ghép 6 năm dự
báo ngoài mẫu ra `gold_model_signals.csv`. Random Forest và Bi-LSTM của nhóm vẫn
giữ, bật thêm bằng `MO_HINH` nếu muốn so sánh.

```
                     OHLCV vàng (khung bất kỳ)
                    /                          \
   NB1 chỉ báo kỹ thuật                    NB2 3 model AI
   luật → dự báo xu hướng                  XGBoost, RF, Bi-LSTM → dự báo xu hướng
                    \                          /
        NB3 chiến lược: xu hướng → lệnh BUY / SELL / FLAT
            → backtest trên CÙNG giai đoạn → Profit, Sharpe, Max DD…
```

NB1 và NB2 chỉ **dự báo xu hướng**: **1 = uptrend (trend tăng) · 0 = sideway ·
−1 = downtrend (trend giảm)** — chưa phải lệnh giao dịch. **NB3** mới áp chiến lược để
đổi xu hướng thành lệnh **BUY / SELL / FLAT** rồi backtest.

NB1 và NB2 **độc lập với nhau**: mỗi notebook tự đọc tệp OHLCV, chạy trước hay sau
đều được. NB3 chạy sau cùng, khi đã có tệp kết quả của cả hai.

Import thư viện và nơi lưu tệp

In [ ]:
import os, json
import pandas as pd
import numpy as np

# ── Nơi trao đổi tệp giữa 3 notebook ─────────────────────────────
# Mỗi notebook Colab chạy trên một máy ảo riêng: tệp NB1/NB2 tạo ra KHÔNG tự có
# mặt ở NB3. Vì vậy cả 3 notebook cùng đọc/ghi vào MỘT thư mục trên Google Drive.
THU_MUC_DRIVE = '/content/drive/MyDrive/Data_NghienCuu'

try:
    from google.colab import files, drive
    TREN_COLAB = True
except ImportError:
    files = drive = None
    TREN_COLAB = False

THU_MUC = '.'
if TREN_COLAB:
    try:
        drive.mount('/content/drive')
        THU_MUC = THU_MUC_DRIVE
    except Exception as loi:
        print('Không gắn được Google Drive (%s).' % loi)
        print('→ Dùng /content: nhớ tải tệp kết quả về và tải lên ở NB3.')
        THU_MUC = '/content'
os.makedirs(THU_MUC, exist_ok=True)
print('Thư mục trao đổi dữ liệu:', os.path.abspath(THU_MUC))


def tim_tep(ten):
    """Tìm tệp đầu vào: thư mục trao đổi → thư mục hiện tại → tải lên (Colab)."""
    for p in (os.path.join(THU_MUC, ten), ten):
        if os.path.exists(p):
            return p
    if TREN_COLAB:
        print('Chưa thấy %s trong %s — hãy tải tệp này lên:' % (ten, THU_MUC))
        up = files.upload()
        if up:
            return list(up.keys())[0]
    raise FileNotFoundError('Không tìm thấy %s. Hãy chạy notebook tạo ra tệp này trước.' % ten)


def luu_tep(bang, ten):
    p = os.path.join(THU_MUC, ten)
    bang.to_csv(p, index=False, encoding='utf-8-sig')
    print('Đã lưu: %s  (%d dòng × %d cột)' % (os.path.abspath(p), bang.shape[0], bang.shape[1]))
    return p

Nhập bộ dữ liệu OHLCV

Để trống `DUONG_DAN_DU_LIEU` thì Colab hiện nút **Choose Files** để tải tệp lên.
Để khỏi tải cùng một tệp hai lần cho NB1 và NB2, có thể đặt tệp vào Drive rồi
điền đường dẫn, ví dụ `/content/drive/MyDrive/Data_NghienCuu/xau_h1.csv`.
Nhận `.csv`, `.txt` (tách bằng dấu phẩy, `;` hoặc tab), `.xlsx`, `.parquet`.

In [ ]:
DUONG_DAN_DU_LIEU = ''

DUONG_DAN_DU_LIEU = DUONG_DAN_DU_LIEU or os.environ.get('NCKH_DU_LIEU', '')
if DUONG_DAN_DU_LIEU:
    file_name = DUONG_DAN_DU_LIEU
elif TREN_COLAB:
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]
else:
    raise ValueError('Đang chạy ngoài Colab: hãy điền DUONG_DAN_DU_LIEU.')

print("Tệp dữ liệu:", file_name)

Đọc dữ liệu

In [ ]:
ten_thuong = file_name.lower()
if ten_thuong.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_name)
elif ten_thuong.endswith(('.parquet', '.pq')):
    df = pd.read_parquet(file_name)
else:
    df = pd.read_csv(file_name)
    # Tệp xuất từ MetaTrader thường tách cột bằng tab hoặc ';' → tự dò lại
    if df.shape[1] == 1:
        df = pd.read_csv(file_name, sep=None, engine='python')

print("Kích thước dữ liệu:", df.shape)
display(df.head())

Chuẩn hóa tên cột và cột thời gian (dùng được cho mọi khung)

Dữ liệu vàng từ các nguồn khác nhau đặt tên cột rất khác nhau. Ô dưới tự nhận
biết mà không cần sửa tay:

- **Tên cột thời gian:** `time`, `datetime`, `date`, `timestamp`, `Gmt time`,
  `<DATE>` + `<TIME>` tách rời (kiểu MetaTrader)…
- **Định dạng thời gian:** `2025-01-02 13:00`, `2025.01.02 13:00`, `02/01/2025`,
  số giây hoặc mili-giây Unix, `20250102`, có hoặc không có múi giờ.
- **Tên cột giá:** `Open/open/<OPEN>/o`, `Close/Adj Close/price`,
  `Volume/Tick Volume/tickvol`…
- **Định dạng số:** `1183.949`, `1183,949`, `1,183.949`, `1.183,949`.

Mọi thời điểm được đưa về **UTC, không kèm múi giờ**. Khung thời gian được suy
ra từ khoảng cách phổ biến nhất giữa hai nến liền nhau.

Ô này **giống hệt nhau ở NB1 và NB2**, nên hai notebook luôn đọc cùng một tệp ra
cùng một bảng dữ liệu.

In [ ]:
def _chuan_ten(c):
    return ' '.join(str(c).strip().lower().replace('<', ' ').replace('>', ' ').replace('_', ' ').split())

# Tên cột theo thứ tự ưu tiên
BI_DANH = {
    'time':   ['datetime', 'date time', 'timestamp', 'time', 'date', 'gmt time', 'local time',
               'time (utc)', 'datetime utc', 'open time', 'opentime', 'thoi gian', 'ngay'],
    'open':   ['open', 'o', 'open price', 'gia mo'],
    'high':   ['high', 'h', 'high price', 'gia cao'],
    'low':    ['low', 'l', 'low price', 'gia thap'],
    'close':  ['close', 'c', 'close price', 'adj close', 'price', 'last', 'gia dong'],
    'volume': ['volume', 'vol', 'tick volume', 'tickvol', 'real volume', 'khoi luong'],
}

def _tim_cot(cac_cot, loai):
    ten = {_chuan_ten(x): x for x in cac_cot}
    for ung_vien in BI_DANH[loai]:
        if ung_vien in ten:
            return ten[ung_vien]
    return None

def _giong_gio(s):
    """Cột chỉ chứa giờ dạng 13:00 hoặc 13:00:00 (cột <TIME> tách rời)."""
    v = s.dropna().astype(str).str.strip().head(200)
    return len(v) > 0 and v.str.fullmatch(r'\d{1,2}:\d{2}(:\d{2})?').mean() > 0.9

def _so(s):
    """Số thực. Chấp nhận 1183.949 · 1183,949 · 1,183.949 · 1.183,949."""
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    s = s.astype(str).str.strip().str.replace(' ', '', regex=False)
    mau = s.head(500)
    # Dấu nào đứng SAU CÙNG là dấu thập phân; dấu còn lại là phân cách hàng nghìn
    phay_la_thap_phan = (mau.str.rfind(',') > mau.str.rfind('.')).mean() > 0.5
    if phay_la_thap_phan:
        s = s.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    else:
        s = s.str.replace(',', '', regex=False)
    return pd.to_numeric(s, errors='coerce')

def _doc_thoi_gian(s):
    """Đọc cột thời gian ở mọi định dạng thường gặp, trả về UTC không múi giờ."""
    if pd.api.types.is_numeric_dtype(s):
        v = pd.to_numeric(s, errors='coerce')
        m = v.dropna().abs().median()
        if 1e7 <= m < 1e8:                                   # dạng 20250102
            return pd.to_datetime(v.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
        don_vi = 'ms' if m > 1e11 else 's'                   # Unix mili-giây hay giây
        return pd.to_datetime(v, unit=don_vi, errors='coerce', utc=True).dt.tz_localize(None)

    s = s.astype(str).str.strip()
    t = pd.to_datetime(s, errors='coerce', utc=True)
    if t.isna().mean() > 0.01:                               # định dạng lẫn lộn → đọc từng dòng
        t = pd.to_datetime(s, errors='coerce', utc=True, format='mixed')
    ung_vien = [t]
    if s.str.contains('/').mean() > 0.5:                     # 02/01/2025: ngày-trước hay tháng-trước?
        ung_vien.append(pd.to_datetime(s, errors='coerce', utc=True, format='mixed', dayfirst=True))
    # Dữ liệu giá luôn xếp theo thời gian: chọn cách đọc ít lỗi nhất và tăng dần nhiều nhất
    diem = lambda x: (x.notna().mean(), (x.diff().dt.total_seconds() > 0).mean())
    return max(ung_vien, key=diem).dt.tz_localize(None)

KHUNG_CHUAN = [(1, 'M1'), (5, 'M5'), (15, 'M15'), (30, 'M30'), (60, 'H1'),
               (240, 'H4'), (1440, 'D1'), (10080, 'W1'), (43200, 'MN')]

def nhan_dien_khung(t):
    phut = t.sort_values().diff().dt.total_seconds().div(60)
    buoc = phut[phut > 0].mode().iloc[0]
    return min(KHUNG_CHUAN, key=lambda k: abs(np.log(k[0] / buoc)))[1], buoc


# ── 1. Cột thời gian
cac_cot = list(df.columns)
ten_chuan = {_chuan_ten(x): x for x in cac_cot}
if 'date' in ten_chuan and 'time' in ten_chuan and _giong_gio(df[ten_chuan['time']]):
    cot_tg = '%s + %s' % (ten_chuan['date'], ten_chuan['time'])
    tho_tg = df[ten_chuan['date']].astype(str).str.strip() + ' ' + df[ten_chuan['time']].astype(str).str.strip()
else:
    cot_tg = _tim_cot(cac_cot, 'time')
    if cot_tg is None:
        raise ValueError('Không tìm thấy cột thời gian trong: %s' % cac_cot)
    tho_tg = df[cot_tg]

ra = pd.DataFrame({'Date': _doc_thoi_gian(tho_tg)})

# ── 2. Cột giá và khối lượng
anh_xa = {}
for loai, ten_moi in [('open', 'Open'), ('high', 'High'), ('low', 'Low'),
                      ('close', 'Close'), ('volume', 'Volume')]:
    cot = _tim_cot(cac_cot, loai)
    anh_xa[ten_moi] = cot
    ra[ten_moi] = _so(df[cot]).values if cot is not None else np.nan

if anh_xa['Close'] is None:
    raise ValueError('Không tìm thấy cột giá đóng cửa trong: %s' % cac_cot)
for ten_moi in ('Open', 'High', 'Low'):
    if anh_xa[ten_moi] is None:
        print('⚠ Thiếu cột %s → tạm dùng giá Close.' % ten_moi)
        ra[ten_moi] = ra['Close']
if anh_xa['Volume'] is None:
    ra['Volume'] = 1.0            # nhiều nguồn Forex không có khối lượng thật

df = ra
KHUNG, BUOC_PHUT = nhan_dien_khung(df['Date'].dropna())

print('Ánh xạ cột:')
print('  %-7s ← %s' % ('Date', cot_tg))
for k, v in anh_xa.items():
    print('  %-7s ← %s' % (k, v if v is not None else '(không có)'))
print('\nKhung thời gian nhận diện: %s  (bước phổ biến %.0f phút)' % (KHUNG, BUOC_PHUT))
print('Giai đoạn: %s → %s' % (df['Date'].min(), df['Date'].max()))
print('Dòng không đọc được thời gian: %d' % df['Date'].isna().sum())

Loại bỏ dữ liệu lỗi và trùng

In [ ]:
print("Trước xử lý:", df.shape)

# Bỏ dòng thiếu thời gian hoặc giá đóng cửa
df = df.dropna(subset=['Date', 'Close'])

# Giá phải lớn hơn 0
df = df[df['Close'] > 0]

# Bỏ nến vi phạm hình học: High < Low, hoặc Close nằm ngoài [Low, High]
hop_le = (df['High'] >= df['Low']) & (df['Close'] <= df['High']) & (df['Close'] >= df['Low'])
print("Nến vi phạm hình học OHLC:", int((~hop_le).sum()))
df = df[hop_le]

# Xóa dòng trùng thời gian, sắp xếp theo thời gian
df = (
    df
    .drop_duplicates(subset=['Date'], keep='last')
    .sort_values('Date')
    .reset_index(drop=True)
)

print("Sau xử lý:", df.shape)
if len(df) < 100:
    raise ValueError('Sau khi làm sạch chỉ còn %d dòng. Kiểm tra lại ánh xạ cột ở ô trên '
                     'và định dạng số của tệp (dấu thập phân, dấu phân cách hàng nghìn).' % len(df))

Tính chỉ báo kỹ thuật (cùng công thức với NB1)

Model dùng **đúng bộ chỉ báo của NB1** (MA, EMA, RSI, MACD, Bollinger, biến động),
tính lại tại đây để NB2 chạy độc lập. Nhờ vậy phép so sánh ở NB3 là công bằng:
**cùng một lượng thông tin**, chỉ khác cách ra quyết định — luật cố định (NB1)
hay model học từ dữ liệu (NB2).

In [ ]:
C = df['Close']
df['Return'] = np.log(C / C.shift(1))
df['MA10'] = C.rolling(window=10).mean()
df['MA30'] = C.rolling(window=30).mean()
df['MA50'] = C.rolling(window=50).mean()
df['EMA12'] = C.ewm(span=12, adjust=False).mean()
df['EMA26'] = C.ewm(span=26, adjust=False).mean()

delta = C.diff()
avg_gain = delta.clip(lower=0).rolling(window=14).mean()
avg_loss = (-delta.clip(upper=0)).rolling(window=14).mean()
df['RSI14'] = 100 - 100 / (1 + avg_gain / avg_loss)

df['MACD'] = df['EMA12'] - df['EMA26']
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']

df['MA20'] = C.rolling(window=20).mean()
df['STD20'] = C.rolling(window=20).std()
df['BB_Upper'] = df['MA20'] + 2 * df['STD20']
df['BB_Lower'] = df['MA20'] - 2 * df['STD20']
df['Volatility20'] = df['Return'].rolling(window=20).std()
print('Đã tính %d chỉ báo.' % (df.shape[1] - 6))

Chuẩn bị đặc trưng cho model

**Mọi chỉ báo có đơn vị USD được chia cho giá đóng cửa.** Vàng đi từ khoảng
1.050 USD (2015) lên hơn 4.500 USD (2025). Model dạng cây không ngoại suy được:
nếu học trên `MA10 = 1.800` rồi gặp `MA10 = 4.000` ở giai đoạn kiểm tra, nó chỉ
trả về vùng giá cao nhất từng thấy. Đổi sang khoảng cách tương đối (ví dụ
`MA10 / Close − 1`) giúp thước đo so sánh được qua mọi mức giá.

In [ ]:
dac_trung = pd.DataFrame(index=df.index)

# Chỉ báo có đơn vị giá → khoảng cách tương đối so với giá đóng cửa
for cot in ['MA10', 'MA30', 'MA50', 'EMA12', 'EMA26', 'MA20', 'BB_Upper', 'BB_Lower']:
    dac_trung['kc_' + cot.lower()] = df[cot] / C - 1
for cot in ['MACD', 'MACD_Signal', 'MACD_Hist', 'STD20']:
    dac_trung[cot.lower() + '_tuong_doi'] = df[cot] / C

# Chỉ báo vốn đã không phụ thuộc mức giá
dac_trung['rsi14'] = df['RSI14'] / 100
dac_trung['return'] = df['Return']
dac_trung['volatility20'] = df['Volatility20']
dac_trung['bb_vi_tri'] = (C - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])

dac_trung = dac_trung.replace([np.inf, -np.inf], np.nan)
feature_cols = list(dac_trung.columns)
print('Số đặc trưng đưa vào model: %d' % len(feature_cols))
print(', '.join(feature_cols))

Walk-forward 2020 → 2025 (cửa sổ huấn luyện mở rộng dần)

Mỗi **năm kiểm tra Y**: huấn luyện trên **toàn bộ dữ liệu trước 01/01/Y**, rồi dự báo cả
năm Y. Sang năm sau, mô hình được huấn luyện lại, có thêm năm vừa qua. Ghép các năm dự báo
lại → **6 năm kiểm tra ngoài mẫu liên tiếp** cho NB3.

| Năm kiểm tra | 2020 | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|---|
| Huấn luyện | 2015 → 2019 | 2015 → 2020 | 2015 → 2021 | 2015 → 2022 | 2015 → 2023 | 2015 → 2024 |

Trong **mỗi lần**: dò lại tham số nhãn `p` chỉ trên tập huấn luyện của lần đó, và bỏ `H`
nến cuối của tập huấn luyện (purging) vì nhãn của chúng đã nhìn sang năm kiểm tra.
Dữ liệu không phủ 2020–2025 → tự chuyển về một lần chia 80 % / 20 %.

In [ ]:
NAM_KIEM_TRA = [2020, 2021, 2022, 2023, 2024, 2025]
MO_HINH = ['xgb']          # thêm 'rf', 'lstm' để chạy cả Random Forest, Bi-LSTM của nhóm (chậm hơn)

nam = df['Date'].dt.year.to_numpy()
FOLD = [(str(y), nam < y, nam == y) for y in NAM_KIEM_TRA
        if (nam < y).sum() > 1000 and (nam == y).sum() > 100]
if not FOLD:
    la_train = np.arange(len(df)) < int(len(df) * 0.8)
    FOLD = [('20% cuối', la_train, ~la_train)]
    print('Dữ liệu không phủ 2020–2025 → một lần chia: 80 % đầu huấn luyện, 20 % cuối kiểm tra.')
print('Walk-forward: %d lần huấn luyện, mô hình %s' % (len(FOLD), MO_HINH))
for ten, tr, te in FOLD:
    print('  Kiểm tra %-9s | huấn luyện %6d nến (%s → %s) | kiểm tra %5d nến'
          % (ten, tr.sum(), df.loc[tr, 'Date'].min().date(), df.loc[tr, 'Date'].max().date(), te.sum()))

Gán nhãn −1 / 0 / 1 — đáp án cho model học

Nhãn dùng phương pháp **Triple Barrier** với rào cản theo **phần trăm giá**:

- Tại nến `t`, đặt rào trên `Close × (1 + p)` và rào dưới `Close × (1 − p)`.
- Quét tối đa `H` nến tiếp theo:
  chạm rào trên trước → **1 (trend tăng)**, chạm rào dưới trước → **−1 (trend giảm)**,
  hết `H` nến mà không chạm, hoặc chạm cả hai trong cùng một nến → **0 (sideway)**.
- `H` nến cuối cùng chưa đủ dữ liệu tương lai nên để trống.

**Tự thích nghi theo khung.** Mỗi khung có biên độ rất khác nhau (M15 đi vài chục
cent, D1 đi vài chục USD), nên `p` không đặt cứng mà được **dò tự động** sao cho
lớp sideway chiếm khoảng 25 %. Việc dò chỉ dùng **tập huấn luyện**.

In [ ]:
# Rào thời gian H (số nến tối đa) theo khung
RAO_THOI_GIAN = {'M1': 60, 'M5': 48, 'M15': 32, 'M30': 24, 'H1': 24,
                 'H4': 12, 'D1': 10, 'W1': 8, 'MN': 6}
SO_NEN_TOI_DA = RAO_THOI_GIAN.get(KHUNG, 24)
TY_LE_SIDEWAY_MUC_TIEU = 0.25
LUOI_PHAN_TRAM = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 1.5,
                  2.0, 3.0, 4.0, 5.0, 7.5, 10.0]


def gan_nhan(cao, thap, dong, p, H):
    """Triple Barrier theo phần trăm giá. p tính theo %, H là số nến tối đa."""
    n = len(dong)
    tren, duoi = dong * (1 + p / 100), dong * (1 - p / 100)
    nhan = np.zeros(n)
    da_xong = np.zeros(n, dtype=bool)
    for k in range(1, H + 1):
        c_k = np.full(n, np.nan); c_k[:n - k] = cao[k:]
        t_k = np.full(n, np.nan); t_k[:n - k] = thap[k:]
        cham_tren, cham_duoi = c_k >= tren, t_k <= duoi
        moi_cham = ~da_xong & (cham_tren | cham_duoi)
        nhan[moi_cham & cham_tren & ~cham_duoi] = 1
        nhan[moi_cham & cham_duoi & ~cham_tren] = -1
        da_xong |= moi_cham               # chạm cả hai trong cùng một nến → giữ 0
    nhan[max(n - H, 0):] = np.nan         # chưa đủ H nến tương lai
    return nhan


cao, thap, dong = df['High'].to_numpy(float), df['Low'].to_numpy(float), df['Close'].to_numpy(float)


def phan_bo(p, n_do):
    """Tỷ lệ tăng / sideway / giảm (%) khi gán nhãn n_do nến đầu với tham số p."""
    nh = gan_nhan(cao[:n_do], thap[:n_do], dong[:n_do], p, SO_NEN_TOI_DA)
    nh = nh[~np.isnan(nh)]
    return 100 * (nh == 1).mean(), 100 * (nh == 0).mean(), 100 * (nh == -1).mean()


def do_p(n_do):
    """Dò p sao cho sideway ≈ mục tiêu, CHỈ trên n_do nến đầu (tập huấn luyện của lần đó).

    Tỷ lệ sideway theo p có dạng CHỮ U, không đơn điệu:
     - p rất nhỏ: nến kế tiếp chạm CẢ HAI rào cùng lúc → gán 0. Đó là nhiễu, không phải sideway.
     - p lớn: hết H nến mà không chạm rào nào → 0 đúng nghĩa sideway.
    Vì vậy chỉ dò trên NHÁNH PHẢI (p lớn hơn điểm đáy), rồi chia đôi khoảng để đạt đúng mục tiêu.
    """
    bang = pd.DataFrame([dict(zip(['p_%', 'tang_%', 'sideway_%', 'giam_%'], (p,) + phan_bo(p, n_do)))
                         for p in LUOI_PHAN_TRAM]).round(2)
    muc_tieu = 100 * TY_LE_SIDEWAY_MUC_TIEU
    i_day = bang['sideway_%'].idxmin()
    nhanh_phai = bang.loc[i_day:]
    vuot = nhanh_phai[nhanh_phai['sideway_%'] >= muc_tieu]
    if vuot.empty:
        return float(nhanh_phai['p_%'].iloc[-1]), bang
    j = vuot.index[0]
    thap_p, cao_p = float(bang.loc[max(j - 1, i_day), 'p_%']), float(bang.loc[j, 'p_%'])
    for _ in range(20):
        giua = (thap_p + cao_p) / 2
        if phan_bo(giua, n_do)[1] < muc_tieu:
            thap_p = giua
        else:
            cao_p = giua
    return round(cao_p, 4), bang


# Minh họa trên tập huấn luyện của lần walk-forward ĐẦU TIÊN
n_dau = int(FOLD[0][1].sum())
P_CHON, bang_do = do_p(n_dau)
print('Khung %s | rào thời gian %d nến | minh họa lần kiểm tra %s: dò trên %d nến huấn luyện'
      % (KHUNG, SO_NEN_TOI_DA, FOLD[0][0], n_dau))
display(bang_do)
tg, sw, gm = phan_bo(P_CHON, n_dau)
print('→ Chọn p = %.4f %%  →  Tăng %.1f %% | Sideway %.1f %% | Giảm %.1f %%  (mục tiêu sideway %.0f %%)'
      % (P_CHON, tg, sw, gm, 100 * TY_LE_SIDEWAY_MUC_TIEU))
print('Mỗi lần walk-forward sẽ dò lại p trên tập huấn luyện của chính lần đó.')

Vùng đệm (purging) — áp dụng trong từng lần walk-forward

Nhãn tại nến `t` nhìn tới `H` nến sau. Vì vậy trong mỗi lần, chỉ những nến huấn luyện
có `t + H` còn nằm **trước** năm kiểm tra mới được dùng; `H` nến sát ranh giới bị loại để
nhãn không "nhìn thấy" năm kiểm tra.

Chuyển đổi xác suất sang dự báo xu hướng ( -1 0 1 ) - Dùng chung cho cả 3 mô hình dưới



In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Ánh xạ nhãn nội bộ để fit vào model (-1 -> 0, 0 -> 1, 1 -> 2)
LABEL_MAP = {-1: 0, 0: 1, 1: 2}
def map_to_internal(y):
    return np.vectorize(LABEL_MAP.get)(y)

def apply_dual_threshold(probs, tau=0.50, delta=0.15):
    """
    Chuyển xác suất 3 kịch bản thành dự báo xu hướng -1 (downtrend), 0 (sideway), 1 (uptrend)
    Cấu hình mặc định đang dùng cho chiến lược Swing (tau=0.50, delta=0.15)
    Cluoc scalping (tau=0.45, delta=0.10) ; Position (tau=0.60, delta=0.20)
    """
    signals = []
    for p in probs:
        p_down, p_flat, p_up = p[0], p[1], p[2]

        # Dự báo UPTREND khi xác suất tăng vượt ngưỡng tuyệt đối và chênh lệch vượt ngưỡng tách biệt
        if p_up >= tau and (p_up - p_down) >= delta:
            signals.append(1)
        # Dự báo DOWNTREND khi xác suất giảm vượt ngưỡng tuyệt đối và chênh lệch vượt ngưỡng tách biệt
        elif p_down >= tau and (p_down - p_up) >= delta:
            signals.append(-1)
        # Nếu lưỡng lự hoặc không đủ tự tin -> SIDEWAY
        else:
            signals.append(0)

    return np.array(signals)

Mô hình XGboost


In [ ]:
def run_xgboost(X_train, y_train, X_test, tau=0.50, delta=0.15):
    y_train_internal = map_to_internal(y_train)

    # Thiết lập siêu tham số chuẩn từ "Nghiên cứu khoa học"
    xgb_model = XGBClassifier(
        max_depth=3,
        learning_rate=0.03,
        n_estimators=150,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective='multi:softprob',
        num_class=3,
        random_state=42,
        n_jobs=-1
    )

    xgb_model.fit(X_train, y_train_internal)

    # Lấy ma trận xác suất và đi qua bộ lọc Ngưỡng kép
    probs = xgb_model.predict_proba(X_test)
    return apply_dual_threshold(probs, tau, delta)

Random forest


In [ ]:
def run_random_forest(X_train, y_train, X_test, tau=0.50, delta=0.15):
    y_train_internal = map_to_internal(y_train)

    rf_model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X_train, y_train_internal)

    probs = rf_model.predict_proba(X_test)
    return apply_dual_threshold(probs, tau, delta)

Bi-LSTM

In [ ]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_size=32, num_layers=1, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(64, 3) # 32 * 2 (bidirectional)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # Lấy trạng thái của nến cuối cùng trong cửa sổ
        # Trả về LOGIT: CrossEntropyLoss đã tự áp softmax bên trong,
        # áp thêm ở đây sẽ thành softmax hai lần và mô hình học rất kém.
        return self.fc(self.dropout(lstm_out[:, -1, :]))

def run_bilstm(X_train, y_train, X_test, seq_len=8, tau=0.50, delta=0.15):
    torch.manual_seed(42)                     # kết quả lặp lại được
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    y_train_int = map_to_internal(y_train)

    def create_seq(X, y=None):
        X_seq, y_seq = [], []
        for i in range(len(X) - seq_len):
            X_seq.append(X[i : i + seq_len])
            if y is not None: y_seq.append(y[i + seq_len])
        return np.array(X_seq), (np.array(y_seq) if y is not None else None)

    X_train_seq, y_train_seq = create_seq(X_train_scaled, y_train_int)
    X_test_seq, _ = create_seq(X_test_scaled)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = BiLSTMModel(X_train.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    loader = DataLoader(TensorDataset(torch.tensor(X_train_seq, dtype=torch.float32),
                                      torch.tensor(y_train_seq, dtype=torch.long)),
                        batch_size=512, shuffle=False)

    model.train()
    for _ in range(8): # 8 epochs
        for b_X, b_y in loader:
            optimizer.zero_grad()
            loss = nn.CrossEntropyLoss()(model(b_X.to(device)), b_y.to(device))
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_test_seq, dtype=torch.float32).to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()

    seq_signals = apply_dual_threshold(probs, tau, delta)

    # Đệm 0 (sideway) cho các nến bị khuyết do thao tác trượt cửa sổ chuỗi
    final_signals = np.zeros(len(X_test), dtype=int)
    final_signals[seq_len:] = seq_signals
    return final_signals

Outputs — huấn luyện walk-forward và ghi tệp dự báo xu hướng cho NB3

Mỗi lần: dò `p` → gán nhãn → lấy nến huấn luyện (bỏ vùng đệm) → huấn luyện → dự báo năm
kiểm tra. Bảng đánh giá ghi **tỉ lệ dự báo trend đúng với nhãn** từng năm — độ chính xác
thuần của mô hình, trước khi đưa vào chiến lược ở NB3.

In [ ]:
# Ngưỡng kép để xác định xu hướng (mặc định cấu hình Swing).
# Scalping: tau=0.45, delta=0.10 | Position: tau=0.60, delta=0.20
tau_config = 0.50
delta_config = 0.15

HAM = {'xgb': ('XGBoost', lambda Xtr, ytr, Xte: run_xgboost(Xtr, ytr, Xte, tau=tau_config, delta=delta_config)),
       'rf': ('Random Forest', lambda Xtr, ytr, Xte: run_random_forest(Xtr, ytr, Xte, tau=tau_config, delta=delta_config)),
       'lstm': ('Bi-LSTM', lambda Xtr, ytr, Xte: run_bilstm(Xtr, ytr, Xte, seq_len=8, tau=tau_config, delta=delta_config))}

du_dac_trung = dac_trung.notna().all(axis=1).to_numpy()
X_all = dac_trung[feature_cols].to_numpy()
cac_phan, danh_gia = [], []
for ten, la_train, la_test in FOLD:
    n_do = int(la_train.sum())                                   # tập huấn luyện là phần đầu chuỗi
    p = do_p(n_do)[0]
    nhan = gan_nhan(cao, thap, dong, p, SO_NEN_TOI_DA)
    i_train = np.flatnonzero(la_train & du_dac_trung & ~np.isnan(nhan))
    i_train = i_train[i_train < n_do - SO_NEN_TOI_DA]            # vùng đệm purging
    i_test = np.flatnonzero(la_test & du_dac_trung)
    X_train, y_train, X_test = X_all[i_train], nhan[i_train].astype(int), X_all[i_test]

    phan = df.iloc[i_test][['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
    phan['label'], phan['fold'], phan['p_nhan_%'] = nhan[i_test], ten, p
    for mh in MO_HINH:
        ten_mh, ham = HAM[mh]
        phan['sig_' + mh] = ham(X_train, y_train, X_test)
        tin, nh = phan['sig_' + mh], phan['label']
        co = tin.ne(0) & nh.notna()
        danh_gia.append({'Năm kiểm tra': ten, 'Mô hình': ten_mh, 'Nến huấn luyện': len(i_train),
                         'p nhãn (%)': p, 'Uptrend': int((tin == 1).sum()), 'Sideway': int((tin == 0).sum()),
                         'Downtrend': int((tin == -1).sum()),
                         'Dự báo trend đúng nhãn (%)': round(100 * (tin[co] == nh[co]).mean(), 2) if co.any() else np.nan})
    cac_phan.append(phan)
    print('  Năm %-9s: huấn luyện %6d nến, p = %.4f %% → %s' % (ten, len(i_train), p,
          ', '.join('%s %s' % (mh, phan['sig_' + mh].value_counts().sort_index().to_dict()) for mh in MO_HINH)))

df_test = pd.concat(cac_phan)
danh_gia = pd.DataFrame(danh_gia)
print('\n--- ĐÁNH GIÁ WALK-FORWARD (tín hiệu: -1 downtrend, 0 sideway, 1 uptrend) ---')
print(danh_gia.to_string(index=False))
luu_tep(danh_gia, 'danh_gia_walk_forward.csv')

# Ghi tệp cho NB3. Cột dự báo xu hướng có tiền tố "sig_" để NB3 nhận đúng.
ra = df_test.rename(columns={'Date': 'time', 'Open': 'open', 'High': 'high',
                             'Low': 'low', 'Close': 'close', 'Volume': 'volume'})
duong_dan_ra = luu_tep(ra, 'gold_model_signals.csv')

# Xem các nến mô hình dự báo có xu hướng (khác sideway)
co_tin = (df_test[['sig_' + mh for mh in MO_HINH]] != 0).any(axis=1)
display(df_test[co_tin][['Date', 'fold', 'label'] + ['sig_' + mh for mh in MO_HINH]].head(15))

In [ ]:
# Tải tệp về máy (không bắt buộc nếu đã lưu trên Google Drive)
if TREN_COLAB and not duong_dan_ra.startswith('/content/drive'):
    files.download(duong_dan_ra)